# Preprocessing Pipeline for Latin

Under each level 2 or smaller heading, choose one or more cells do run. Each cell is a single step that can be added to the whole pipeline. To skip a step, simply convert the cell to `raw` type with `esc` + `r` keys. By default, all steps are initially skipped.

## Initial Setup

Frequently used Python modules.

In [18]:
import re, os, cltk, requests, spacy, textblob
from pprint import pprint
from termcolor import colored, cprint
import unicodedata
import json

In [2]:
from importlib.metadata import version

print("CLTK version:", version("cltk"))

CLTK version: 2.5.1


### *From URI*

In [3]:
def load_uri(uri: str) -> str:
    response = requests.get(uri, timeout=30)
    response.raise_for_status()
    return response.text

In [4]:
api_url = (
    "https://api.github.com/repos/"
    "PerseusDL/canonical-latinLit/git/trees/master?recursive=1"
)

response = requests.get(api_url, timeout=60)
response.raise_for_status()

files = response.json()["tree"]

matches = [
    item["path"]
    for item in files
    if item["path"].endswith(".xml")
    and "/phi0448/" in item["path"]
    and "lat" in item["path"].lower()
]

print("Matching Latin editions:")
for path in matches:
    print(path)

Matching Latin editions:
data/phi0448/phi001/phi0448.phi001.perseus-lat2.xml
data/phi0448/phi002/phi0448.phi002.perseus-lat2.xml
data/phi0448/phi002/phi0448.phi002.perseus-lat3.xml


In [5]:
def preprocess_latin(text: str) -> str:
    # Normalize Unicode
    text = unicodedata.normalize("NFC", text)

    # Replace Latin ligatures
    text = text.replace("æ", "ae").replace("Æ", "Ae")
    text = text.replace("œ", "oe").replace("Œ", "Oe")

    # Remove editorial annotations in angle brackets
    text = re.sub(r"<[^>]*>", "", text)

    # Remove editorial annotations in square brackets
    text = re.sub(r"\[[^\]]*\]", "", text)

    # Rejoin words split across lines
    text = re.sub(r"(\w)-\s*\n\s*(\w)", r"\1\2", text)

    # Normalize spaces while preserving paragraph boundaries
    text = re.sub(r"[^\S\n]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

In [6]:
file_path = "data/phi0448/phi001/phi0448.phi001.perseus-lat2.xml"

url = (
    "https://raw.githubusercontent.com/"
    "PerseusDL/canonical-latinLit/master/"
    + file_path
)

response = requests.get(url, timeout=60)
response.raise_for_status()

raw_xml = response.text

print("Downloaded characters:", len(raw_xml))
print(raw_xml[:500])

Downloaded characters: 691979
<?xml version="1.0" encoding="UTF-8"?>
<?xml-model href="https://epidoc.stoa.org/schema/latest/tei-epidoc.rng" schematypens="http://relaxng.org/ns/structure/1.0"?>
<TEI xmlns="http://www.tei-c.org/ns/1.0">
<teiHeader xml:lang="eng">
<fileDesc>
<titleStmt>
<title xml:lang="lat">De bello Gallico</title>
<author>Julius Caesar</author>
<editor>T. Rice Holmes</editor>
<sponsor>Perseus Project, Tufts University</sponsor>
<principal>Gregory Crane</principal>
<respStmt>
<resp>Prepared under the supervis


In [7]:
import xml.etree.ElementTree as ET

root = ET.fromstring(raw_xml)

ns = {"tei": "http://www.tei-c.org/ns/1.0"}

body = root.find(".//tei:text/tei:body", ns)

if body is None:
    raise ValueError("Could not find the TEI text body")

latin_text = " ".join(
    text.strip()
    for text in body.itertext()
    if text.strip()
)

print("Extracted characters:", len(latin_text))
print(latin_text[:1000])

Extracted characters: 376375
COMMENTARIUS PRIMUS Gallia est omnis divisa in partes tres, quarum unam incolunt Belgae, aliam Aquitani, tertiam qui ipsorum lingua Celtae, nostra Galli appellantur. Hi omnes lingua, institutis, legibus inter se differunt. Gallos ab Aquitanis Garumna flumen, a Belgis Matrona et Sequana dividit. Horum omnium fortissimi sunt Belgae, propterea quod a cultu atque humanitate provinciae longissime absunt, minimeque ad eos mercatores saepe commeant atque ea quae ad effeminandos animos pertinent important, proximique sunt Germanis, qui trans Rhenum incolunt, quibuscum continenter bellum gerunt. Qua de causa Helvetii quoque reliquos Gallos virtute praecedunt, quod fere cotidianis proeliis cum Germanis contendunt, cum aut suis finibus eos prohibent aut ipsi in eorum finibus bellum gerunt. [Eorum una pars, quam Gallos obtinere dictum est, initium capit a flumine Rhodano, continetur Garumna flumine, Oceano, finibus Belgarum, attingit etiam ab Sequanis et Helvetiis flum

In [8]:
from collections import Counter

ns_uri = "http://www.tei-c.org/ns/1.0"

divisions = root.findall(".//tei:body//tei:div", ns)

print("Total divisions:", len(divisions))

print("\nDivision types:")
print(Counter(div.get("type") for div in divisions))

print("\nFirst 15 divisions:")
for div in divisions[:15]:
    print({
        "type": div.get("type"),
        "subtype": div.get("subtype"),
        "n": div.get("n"),
        "id": div.get("{http://www.w3.org/XML/1998/namespace}id")
    })

Total divisions: 2563

Division types:
Counter({'textpart': 2562, 'edition': 1})

First 15 divisions:
{'type': 'edition', 'subtype': None, 'n': None, 'id': None}
{'type': 'textpart', 'subtype': 'book', 'n': '1', 'id': None}
{'type': 'textpart', 'subtype': 'chapter', 'n': '1', 'id': None}
{'type': 'textpart', 'subtype': 'section', 'n': '1', 'id': None}
{'type': 'textpart', 'subtype': 'section', 'n': '2', 'id': None}
{'type': 'textpart', 'subtype': 'section', 'n': '3', 'id': None}
{'type': 'textpart', 'subtype': 'section', 'n': '4', 'id': None}
{'type': 'textpart', 'subtype': 'section', 'n': '5', 'id': None}
{'type': 'textpart', 'subtype': 'section', 'n': '6', 'id': None}
{'type': 'textpart', 'subtype': 'section', 'n': '7', 'id': None}
{'type': 'textpart', 'subtype': 'chapter', 'n': '2', 'id': None}
{'type': 'textpart', 'subtype': 'section', 'n': '1', 'id': None}
{'type': 'textpart', 'subtype': 'section', 'n': '2', 'id': None}
{'type': 'textpart', 'subtype': 'section', 'n': '3', 'id': No

In [9]:
import pandas as pd

ns = {"tei": "http://www.tei-c.org/ns/1.0"}

records = []

for book in root.findall(".//tei:div[@subtype='book']", ns):
    book_num = book.get("n")

    for chapter in book.findall("./tei:div[@subtype='chapter']", ns):
        chapter_num = chapter.get("n")

        for section in chapter.findall("./tei:div[@subtype='section']", ns):
            section_num = section.get("n")

            text = " ".join(
                fragment.strip()
                for fragment in section.itertext()
                if fragment.strip()
            )

            if text:
                records.append({
                    "work": "De Bello Gallico",
                    "book": book_num,
                    "chapter": chapter_num,
                    "section": section_num,
                    "text": text
                })

df = pd.DataFrame(records)

print("Total sections:", len(df))
display(df.head())

Total sections: 2150


,work,book,chapter,section,text
0,De Bello Gallico,1,1,1,"Gallia est omnis divisa in partes tres, quarum..."
1,De Bello Gallico,1,1,2,"Hi omnes lingua, institutis, legibus inter se ..."
2,De Bello Gallico,1,1,3,"Horum omnium fortissimi sunt Belgae, propterea..."
3,De Bello Gallico,1,1,4,"proximique sunt Germanis, qui trans Rhenum inc..."
4,De Bello Gallico,1,1,5,"[Eorum una pars, quam Gallos obtinere dictum e..."


In [10]:
df["word_count"] = df["text"].str.split().str.len()

print(df["word_count"].describe())
print("\nLongest sections:")
display(df.nlargest(5, "word_count"))

count    2150.000000
mean       23.860465
std        10.890464
min         3.000000
25%        16.000000
50%        22.000000
75%        30.000000
max       106.000000
Name: word_count, dtype: float64

Longest sections:


,work,book,chapter,section,text,word_count
422,De Bello Gallico,2,25,1,Caesar ab X. legionis cohortatione ad dextrum ...,106
105,De Bello Gallico,1,20,5,Haec cum pluribus verbis flens a Caesare peter...,79
140,De Bello Gallico,1,28,3,"Helvetios, Tulingos, Latobrigos in fines suos,...",71
388,De Bello Gallico,2,17,4,Adiuvabat etiam eorum consilium qui rem defere...,71
970,De Bello Gallico,5,28,3,Lucius Aurunculeius compluresque tribuni milit...,70


In [11]:
from transformers import AutoTokenizer

MODEL_NAME = "latincy/latin-bert"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=False,
    trust_remote_code=True
)

df["token_count"] = df["text"].apply(
    lambda text: len(
        tokenizer.encode(text, add_special_tokens=True)
    )
)

print(df["token_count"].describe())

print("\nSections exceeding 512 tokens:")
print((df["token_count"] > 512).sum())

Could not reach the Hub ([WinError 10054] An existing connection was forcibly closed by the remote host). Using cached commit hash for 'latincy/latin-bert'.


count    2150.000000
mean       37.322791
std        16.280292
min         8.000000
25%        25.000000
50%        35.000000
75%        46.000000
max       160.000000
Name: token_count, dtype: float64

Sections exceeding 512 tokens:
0


In [12]:
from pathlib import Path

output_dir = Path("../../SentimentAnalysisLatin/data/raw")
output_dir.mkdir(parents=True, exist_ok=True)

df["cts_urn"] = df.apply(
    lambda row: (
        "urn:cts:latinLit:phi0448.phi001.perseus-lat2:"
        f"{row['book']}.{row['chapter']}.{row['section']}"
    ),
    axis=1
)

df.to_csv(
    output_dir / "de_bello_gallico_sections.csv",
    index=False,
    encoding="utf-8"
)

print(f"Saved {len(df)} sections.")

Saved 2150 sections.


In [20]:
MODEL_NAME = "qwen2.5:7b"
OLLAMA_URL = "http://localhost:11434/api/chat"

SYSTEM_PROMPT = """
You are an expert in Classical Latin and literary sentiment analysis.

Read the original Latin passage directly. Do not translate it into English.

Classify its expressed emotional tone:
- negative: predominantly fear, grief, anger, suffering, hostility, or distress
- neutral: predominantly factual, descriptive, or emotionally balanced
- positive: predominantly joy, hope, admiration, affection, relief, or celebration

Judge the emotional tone conveyed by the passage, not whether
its events are morally good or bad.

If sentiment is mixed or unclear, prefer neutral.

Return only a JSON object with:
{
  "sentiment": "negative|neutral|positive",
  "confidence": "low|medium|high",
  "reason": "A brief explanation in English"
}
"""

def label_latin(text):
    response = requests.post(
        OLLAMA_URL,
        json={
            "model": MODEL_NAME,
            "stream": False,
            "format": "json",
            "options": {
                "temperature": 0,
                "num_ctx": 4096
            },
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": text}
            ]
        },
        timeout=180
    )

    response.raise_for_status()

    result = json.loads(response.json()["message"]["content"])

    if result["sentiment"] not in {"negative", "neutral", "positive"}:
        raise ValueError(f"Invalid sentiment: {result}")

    return result

In [28]:
OUTPUT_PATH = Path("../../SentimentAnalysisLatin/data/labeled/caesar_pseudo_labels.csv")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

label_map = {
    "negative": 0,
    "neutral": 1,
    "positive": 2
}

# Resume existing progress, or initialize the dataset
if OUTPUT_PATH.exists():
    labeled_df = pd.read_csv(OUTPUT_PATH)
else:
    labeled_df = df.copy()

    labeled_df["sentiment"] = None
    labeled_df["confidence"] = None
    labeled_df["reason"] = None
    labeled_df["label"] = None
    labeled_df["teacher_model"] = MODEL_NAME
    labeled_df["prompt_version"] = "latin_sentiment_v1"

# Label each passage
for i in range(len(labeled_df)):

    # Skip already labeled passages
    if pd.notna(labeled_df.at[i, "sentiment"]):
        continue

    try:
        result = label_latin(labeled_df.at[i, "text"])

        labeled_df.at[i, "sentiment"] = result["sentiment"]
        labeled_df.at[i, "confidence"] = result["confidence"]
        labeled_df.at[i, "reason"] = result["reason"]
        labeled_df.at[i, "label"] = label_map[result["sentiment"]]

        if (i + 1) % 25 == 0:
            labeled_df.to_csv(OUTPUT_PATH, index=False)
            print(f"Processed {i + 1}/{len(labeled_df)}")

    except Exception as e:
        print(f"Error at section {i}: {e}")
        break

# Save remaining results
labeled_df.to_csv(OUTPUT_PATH, index=False)

print("\nLabeling stopped or completed.")
print("Labeled:", labeled_df["sentiment"].notna().sum())
print("Total:", len(labeled_df))

Processed 25/2150
Processed 50/2150
Processed 75/2150
Processed 100/2150
Processed 125/2150
Processed 150/2150
Processed 175/2150
Processed 200/2150
Processed 225/2150
Processed 250/2150
Processed 275/2150
Processed 300/2150
Processed 325/2150
Processed 350/2150
Processed 375/2150
Processed 400/2150
Processed 425/2150
Processed 450/2150
Processed 475/2150
Processed 500/2150
Processed 525/2150
Processed 550/2150
Processed 575/2150
Processed 600/2150
Processed 625/2150
Processed 650/2150
Processed 675/2150
Processed 700/2150
Processed 725/2150
Processed 750/2150
Processed 775/2150
Processed 800/2150
Processed 825/2150
Processed 850/2150
Processed 875/2150
Processed 900/2150
Processed 925/2150
Processed 950/2150
Processed 975/2150
Processed 1000/2150
Processed 1025/2150
Processed 1050/2150
Processed 1075/2150
Processed 1100/2150
Processed 1125/2150
Processed 1150/2150
Processed 1175/2150
Processed 1200/2150
Processed 1225/2150
Processed 1250/2150
Processed 1275/2150
Processed 1300/2150
Pr